In [ ]:
import numpy as np
import os

In [ ]:
command_qmasm = 'qmasm --format="numpy" -o="out.npz" --pin="Query.Valid := true" --solver="tabu" work/animal_unfold.qmasm'
command_QAP = 'QA-Prolog --qmasm-args="--asd" --work-dir="/home/dadi/Thesis/Code/2_QA-prolog/3_QA-prolog/KB2/work"'
query = 'subClass(mammal, animal).'
file = 'animal_unfold.pl'

In [ ]:
def read_file(file_name):
    with open(file_name, 'r') as file_in:
        lines = file_in.readlines()
    return lines

In [ ]:
def write_unfold_calls(file_out, n_unfold):
    file_out.write('subClass0(C0,C1) :- \n\tsubclass(C0, C1).\n')
    for i in range(1, n_unfold):
        rule = 'subClass'+ str(i) + '(C0,C2) :-\n'      +\
               '\tsubClass' + str(i-1) + '(C0,C3),\n'   +\
               '\tsubclass(C3, C2).\n'
        file_out.write(rule)
    for i in range(n_unfold):
        file_out.write('subClass(C0,C1) :- subClass' + str(i) +'(C0, C1).\n')

In [ ]:
def write_file(file_name_in, file_name_out, n_unfold):
    t_box = read_file(file_name_in)
    file_out = open(file_name_out, 'w')
    file_out.writelines(t_box)
    file_out.write('\n\n')
    write_unfold_calls(file_out, n_unfold)
    file_out.close()
    

In [ ]:
def run_experiments(file_name_in, file_name_out, n_unfold_max):
    n_qubit = []
    for i in range (2, n_unfold_max + 1):
        write_file(file_name_in, file_name_out, i)
        command = command_QAP + ' --query="' + query + '" ' + file_name_out
        print(command)
        #os.system('cat ' + file_name_out)
        os.system(command)
        os.system(command_qmasm)
        data = np.load("out.npz")
        print(len(data["syms"]))
        n_qubit.append(len(data["syms"]))
    return n_qubit

In [ ]:
n_qubit = run_experiments('animal.pl', 'animal_unfold.pl', 3)

In [ ]:
for i, n in enumerate(n_qubit):
    print(i + 1,", ", sep="", end="")
print()
for i, n in enumerate(n_qubit):
    print(n,", ", sep="", end="")